# 09 — Classifier Validation Comparison

All conclusions here use validation only. The active discovery root is exclusively `results/publication_v2/`; V1 and the retired classifier matrix are ignored.

## 1. Discover the 24 experiment outputs

In [ ]:
from pathlib import Path
import json
import sys
ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'configs').is_dir())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'notebooks/utility'))
from notebooks.utility.classifier_analysis import (build_all_validation_ensembles, compare_validation,
    discover_experiments, ensemble_metric_table, plot_ensemble_curves, plot_ensemble_overview)
inventory = discover_experiments(ROOT)
sum(row['complete'] for row in inventory), inventory

## 2. Completeness

In [ ]:
missing_jobs = [row for row in inventory if not row['complete']]
missing_jobs

## 3. Build the eight three-seed ensembles

In [ ]:
BUILD_ENSEMBLES = False
if BUILD_ENSEMBLES and missing_jobs:
    raise RuntimeError(f'Cannot build 8 ensembles: {len(missing_jobs)} of 24 seed runs are incomplete')
ensembles = build_all_validation_ensembles(ROOT) if BUILD_ENSEMBLES else None
ensembles if ensembles is not None else 'Not yet evaluated — set BUILD_ENSEMBLES=True after all 24 validation outputs exist.'

## 4. Alignment checks

In [ ]:
alignment_requirements = ['identical patient/image keys', 'identical labels', 'no duplicates', 'no missing rows', 'finite probabilities', 'same validation manifest']
alignment_requirements

## 5. Patient-level aggregation

In [ ]:
patient_policy = 'mean image probability within patient; patient is the bootstrap unit'
patient_policy

## 6. Metrics and confidence intervals

In [ ]:
ensemble_table = ensemble_metric_table(ensembles) if ensembles is not None else None
ensemble_table if ensemble_table is not None else 'Not yet evaluated'

## 7. Variability across seeds

In [ ]:
seed_metrics = ({(row['architecture'], row['condition']): row['seed_metrics'] for row in ensembles}
                if ensembles is not None else None)
seed_metrics if seed_metrics is not None else 'Not yet evaluated'

## 8. Four-condition comparisons

In [ ]:
RUN_COMPARISONS = BUILD_ENSEMBLES
comparison = compare_validation(ROOT, ensembles) if RUN_COMPARISONS else None
comparison or 'Not yet evaluated — this is validation analysis; Holm correction covers the eight preregistered patient-level PR-AUC comparisons.'

## 9. Validation plots and tables

In [ ]:
overview_figure = plot_ensemble_overview(ensembles) if ensembles is not None else None
curve_figure = plot_ensemble_curves(ROOT, ensembles) if ensembles is not None else None
(overview_figure, curve_figure) if overview_figure else 'Not yet evaluated — plots include PR-AUC intervals, ROC-AUC/Brier/ECE table, deltas vs real_only, seed mean/SD, heatmap, PR/ROC and calibration curves.'

## 10. Validation-only conclusions

Do not infer definitive final-evaluation performance here and do not revise models or thresholds using any historical test result.